# 06 Evaluation

In [ ]:
# %pip install -q "chromadb>=1,<2" sentence-transformers openai python-dotenv pandas numpy matplotlib tiktoken "mlflow>=3,<4"

## 1. Configuration

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd()
BASE = ROOT.parent if ROOT.name == 'notebooks' else ROOT
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
from src.evaluation_dataset import make_dataset
from src.evaluation import run_evaluation

RUN_LIVE_EVALUATION = False
API_BUDGET_USD = 5.0
WORKERS = 3
REPORTS = BASE / 'reports'
EVALUATION = REPORTS / 'evaluation'
print('Project:', BASE)
print('Live API calls enabled:', RUN_LIVE_EVALUATION)

## 2. Verify steps 4 and 5

In [ ]:
verification_path = REPORTS / 'step_4_5_verification.md'
if verification_path.exists():
    display(Markdown(verification_path.read_text(encoding='utf-8')))
else:
    print('Verification report not found. Inspect notebook 4/5 source and saved results before evaluation.')

## 3. Thirty-question test set and shared references

In [ ]:
dataset = make_dataset()
test_table = pd.DataFrame(dataset)
display(test_table[['id','question_type','banks','risk','question']])
display(test_table.groupby('question_type').size().to_frame('questions'))
assert len(dataset) == 30 and len({q['id'] for q in dataset}) == 30

## 4. Generate, judge and track

In [ ]:
if RUN_LIVE_EVALUATION:
    evaluation = run_evaluation(BASE, live=True, workers=WORKERS, budget_usd=API_BUDGET_USD)
    RUN_PATH = evaluation['run_path']
else:
    pointer = EVALUATION / 'latest_run.json'
    if not pointer.exists():
        raise FileNotFoundError('No saved evaluation. Set RUN_LIVE_EVALUATION=True to run generation and judging.')
    latest = json.loads(pointer.read_text(encoding='utf-8'))
    RUN_PATH = EVALUATION / latest['run_id']
    print('Loading saved evaluation; no generation/judge calls made.')

manifest = json.loads((RUN_PATH/'manifest.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(RUN_PATH/'master_comparison.csv')
metrics = pd.read_csv(RUN_PATH/'per_question_metrics.csv')
cases = [json.loads(path.read_text(encoding='utf-8')) for path in sorted((RUN_PATH/'cases').glob('*.json'))]
print('Evaluation:', manifest['run_id'])
print('Reports:', RUN_PATH)
print('Scored cases:', int(metrics['status'].eq('complete').sum()), '/ 90')
display(comparison)

## 5. Master comparison and quality thresholds

In [ ]:
quality_columns = [c for c in ['config','scored_questions','complete','faithfulness','relevance',
    'completeness','hallucination_risk_count','meets_quality_target'] if c in comparison]
display(comparison[quality_columns])
if (RUN_PATH/'quality_comparison.png').exists():
    display(Image(filename=str(RUN_PATH/'quality_comparison.png')))
for dimension in ['question_type','risk']:
    scored = metrics[metrics['status']=='complete']
    if not scored.empty:
        display(scored.groupby(['config',dimension])[['faithfulness','relevance','completeness']].mean())

## 6. Hallucination analysis and source inspection

In [ ]:
hallucinations = json.loads((RUN_PATH/'hallucination_cases.json').read_text(encoding='utf-8'))
print('Flagged grounding-risk cases:', len(hallucinations))
for case in hallucinations:
    print(case['config'], case['question']['id'], case['question']['question'])
    print(case['judgment']['faithfulness_reason'])
display(Markdown((RUN_PATH/'hallucination_analysis.md').read_text(encoding='utf-8')))

# Choose one record to inspect. Keep the same case ID when recording a human review.
CASE_CONFIG, QUESTION_ID = 'agentic', 'eval_01'
selected = next((c for c in cases if c['config']==CASE_CONFIG and c['question']['id']==QUESTION_ID), None)
if selected and 'generation' in selected:
    print('QUESTION:', selected['question']['question'])
    print('ANSWER:', selected['generation']['answer'])
    display(selected.get('judgment', {}))
    for hit in selected['generation']['sources']:
        print('\n', hit['chunk_id'], hit.get('source_url'), hit.get('page_start'), hit.get('page_end'))
        print(hit['text'])

## 7. Cost per query and annual projection

In [ ]:
cost_columns = [c for c in ['config','mean_embedding_cost_usd','mean_retrieval_cost_usd',
    'mean_generation_cost_usd','mean_cost_per_query_usd','annual_cost_1000_per_day_usd',
    'judge_total_cost_usd','mean_latency_seconds','p95_latency_seconds'] if c in comparison]
display(comparison[cost_columns])
display(manifest['prices_per_million_usd'])
if (RUN_PATH/'cost_latency.png').exists():
    display(Image(filename=str(RUN_PATH/'cost_latency.png')))
print('API ledger:', EVALUATION/'api_usage.jsonl')

## 8. MLflow experiments

Each complete configuration has a tracked run in `models/mlflow.db`, including quality/cost/latency metrics, settings, source/test hashes, rubric and case artifacts.

In [ ]:
tracking_file = RUN_PATH/'mlflow_runs.json'
if tracking_file.exists():
    display(pd.DataFrame(json.loads(tracking_file.read_text(encoding='utf-8'))))
else:
    print('No complete configuration logged yet. Complete the run before claiming MLflow comparison.')